# Serial Position Curve Analysis

This notebook analyzes behavioral data from a directed forgetting experiment.

**Experiment design:**
- Participants studied two lists of words in each trial
- After the first list, they received either a "Remember" or "Forget" cue
- After both lists, participants attempted to recall all words

**Serial position curves** show the probability of recalling a word as a function of its position in the study list.

In [ ]:
# Import required packages
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

## Load behavioral data

In [ ]:
# Load the behavioral data
with open('behavioral_data.pkl', 'rb') as f:
    behavior = pickle.load(f)

print(f"Loaded data for {len(behavior)} participants")
print(f"\nParticipant IDs: {list(behavior.keys())}")

## Explore the data structure

In [ ]:
# Look at one participant's data
example_subj = list(behavior.keys())[0]
print(f"Example participant: {example_subj}")
print(f"\nAvailable data fields:")
for key in sorted(behavior[example_subj].keys()):
    print(f"  - {key}")

In [ ]:
# Key fields:
# - spc: serial position curve (lists x positions matrix)
# - cuetype: 0 = Remember cue, 1 = Forget cue
# - recmats: recall matrices showing which words were recalled

print(f"Serial position curve shape: {behavior[example_subj]['spc'].shape}")
print(f"Cue types shape: {behavior[example_subj]['cuetype'].shape}")
print(f"\nExample cue types (first 10 lists): {behavior[example_subj]['cuetype'][:10].ravel()}")
print("  0 = Remember, 1 = Forget")

## Single participant serial position curve

In [ ]:
# Create a dataframe for plotting
spc = pd.DataFrame(behavior[example_subj]['spc']).reset_index().rename({'index': 'List'}, axis=1)
spc = spc.melt(id_vars='List', var_name='Serial position', value_name='Recall probability')
spc['Cue'] = spc['List'].apply(lambda x: 'Remember' if behavior[example_subj]['cuetype'][x] == 0 else 'Forget')

# Convert serial position to 1-indexed
spc['Serial position'] = spc['Serial position'] + 1

# Plot
plt.figure(figsize=(10, 6))
sns.lineplot(data=spc, x='Serial position', y='Recall probability', 
             hue='Cue', palette='Greens', errorbar=('ci', 50))
plt.title(f'Serial Position Curve: {example_subj}', fontsize=16)
plt.xlabel('Serial Position', fontsize=14)
plt.ylabel('Recall Probability', fontsize=14)
plt.legend(loc='lower right', fontsize=12)
sns.despine(top=True, right=True)
plt.tight_layout()
plt.show()

## Aggregate serial position curves across all participants

In [ ]:
# Combine data from all participants
all_spc = []

for subj_id, subj_data in behavior.items():
    spc_df = pd.DataFrame(subj_data['spc']).reset_index().rename({'index': 'List'}, axis=1)
    spc_df = spc_df.melt(id_vars='List', var_name='Serial position', value_name='Recall probability')
    spc_df['Cue'] = spc_df['List'].apply(lambda x: 'Remember' if subj_data['cuetype'][x] == 0 else 'Forget')
    spc_df['Participant'] = subj_id
    spc_df['Serial position'] = spc_df['Serial position'] + 1
    all_spc.append(spc_df)

all_spc = pd.concat(all_spc, ignore_index=True)

print(f"Combined data shape: {all_spc.shape}")
print(f"\nFirst few rows:")
all_spc.head()

In [ ]:
# Plot aggregate serial position curves
plt.figure(figsize=(12, 7))
sns.lineplot(data=all_spc, x='Serial position', y='Recall probability',
             hue='Cue', palette='Greens', errorbar='se', linewidth=2.5)
plt.title('Serial Position Curves Across All Participants', fontsize=18, fontweight='bold')
plt.xlabel('Serial Position', fontsize=16)
plt.ylabel('Recall Probability', fontsize=16)
plt.legend(title='Cue Type', loc='lower right', fontsize=14, title_fontsize=14)
plt.grid(alpha=0.3, linestyle='--')
sns.despine(top=True, right=True)
plt.tight_layout()
plt.show()

## Summary statistics

In [ ]:
# Calculate overall recall performance by cue type
summary = all_spc.groupby(['Participant', 'Cue'])['Recall probability'].mean().reset_index()
summary_stats = summary.groupby('Cue')['Recall probability'].agg(['mean', 'std', 'sem'])

print("Overall recall performance by cue type:")
print(summary_stats)
print("\n(sem = standard error of the mean)")

In [ ]:
# Visualize overall recall by cue type
plt.figure(figsize=(8, 6))
sns.barplot(data=summary, x='Cue', y='Recall probability', 
            palette='Greens', errorbar='se')
plt.title('Overall Recall Performance by Cue Type', fontsize=16, fontweight='bold')
plt.xlabel('Cue Type', fontsize=14)
plt.ylabel('Mean Recall Probability', fontsize=14)
plt.ylim(0, max(summary['Recall probability']) * 1.2)
sns.despine(top=True, right=True)
plt.tight_layout()
plt.show()

## Recall by serial position region

In [ ]:
# Define early (first 3 positions), middle, and late (last 3 positions) regions
max_position = all_spc['Serial position'].max()
early_positions = [1, 2, 3]
late_positions = [max_position - 2, max_position - 1, max_position]
middle_positions = list(range(4, max_position - 2))

def classify_position(pos):
    if pos in early_positions:
        return 'Early (1-3)'
    elif pos in late_positions:
        return f'Late ({max_position-2}-{max_position})'
    else:
        return 'Middle'

all_spc['Position region'] = all_spc['Serial position'].apply(classify_position)

# Calculate means for each region
position_effects = all_spc.groupby(['Participant', 'Cue', 'Position region'])['Recall probability'].mean().reset_index()

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(data=position_effects, x='Position region', y='Recall probability',
            hue='Cue', palette='Greens', errorbar='se')
plt.title('Recall by Serial Position Region and Cue Type', fontsize=16, fontweight='bold')
plt.xlabel('List Region', fontsize=14)
plt.ylabel('Mean Recall Probability', fontsize=14)
plt.legend(title='Cue Type', fontsize=12, title_fontsize=12)
sns.despine(top=True, right=True)
plt.tight_layout()
plt.show()